# Урок 5.5. Мониторинг и алертинг для локальных LLM

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 5 из 7

В уроке 3.7 мы делали сервис наблюдаемым: структурные логи, метрики руками,
перцентили. В уроке 4.3 — подключали Prometheus и трейсинг. Всё это работает
и здесь. Но у локальной LLM есть особенность, которой нет у облачного API:
**она болеет железом**. Облачный API мониторят снаружи — латентность, ошибки,
лимиты. Локальная модель добавляет три этажа своих проблем: хост (RAM, swap,
троттлинг), демон (жив ли, что загружено) и сама генерация (скорость, холодные
старты). Сегодня строим наблюдение за всеми тремя.

## 🎯 Что вы узнаете

- Три уровня health-check — от «пинга» до настоящей генерации — и цену каждого
- Как собирать метрики скорости не пробами, а с каждого боевого ответа Ollama
- Как выглядит главная деградация локальной LLM — «модель тихо съехала с GPU» — и как её ловить
- Правило for-duration: почему алерт не должен срабатывать на единичный выброс
- Как оформить всё это в сторожевой скрипт и метрики Prometheus


## Три этажа наблюдения

```
Этаж 3. КАЧЕСТВО СЕРВИСА     скорость ток/с, холодные старты, доля отказов
        (уроки 3.3, 3.7)     -> метрики из каждого ответа + редкие глубокие пробы
                 |
Этаж 2. ДЕМОН OLLAMA         жив? какая версия? что в памяти? на GPU или CPU?
        (урок 5.3)           -> /api/version, /api/tags, /api/ps
                 |
Этаж 1. ХОСТ                 свободная RAM, swap, загрузка CPU
        (этот урок)          -> psutil
```

Облачному API вы доверяете этажи 1–2 (это проблемы провайдера) и мониторите
только третий. Со своим железом вы отвечаете за все три — и, по опыту уроков
3.2 и 5.3, ломаются они по-настоящему: Ollama самообновлялась посреди работы,
контейнеры пропадали при перезапуске Docker.

## Пирамида health-check: три уровня и их цена

| Уровень | Что проверяет | Чего НЕ видит | Цена |
|---|---|---|---|
| L1 `/api/version` | процесс жив, порт отвечает | модель может не грузиться вовсе | миллисекунды |
| L2 `/api/tags` + `/api/ps` | модель скачана; что в памяти и **где** (GPU/CPU) | генерация может висеть | миллисекунды |
| L3 генерация N токенов | весь путь: загрузка → prefill → генерация | — | секунды; **грузит модель!** |

Ключевая мысль: **L3 — деструктивная проверка**. Она сама загружает модель
(отбирая RAM), сбрасывает таймер keep_alive и занимает слот параллелизма —
при `NUM_PARALLEL=1` настоящий пользовательский запрос будет ждать вашу пробу.
Наблюдатель влияет на систему. Поэтому L1/L2 можно дёргать хоть каждые 10 секунд,
а L3 — раз в несколько минут.

Измерим цену каждого уровня (модель предварительно выгрузим, чтобы L3 честно
показал холодный путь):


In [ ]:
import time
from collections import deque

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"  # не localhost — урок 5.3
NS = 1_000_000_000
MODEL = "qwen2.5:3b"


def ollama_available() -> bool:
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


def unload_model(model: str = MODEL) -> None:
    httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0}, timeout=60)


OLLAMA_UP = ollama_available()
if OLLAMA_UP:
    unload_model()

    t0 = time.perf_counter()
    version = httpx.get(f"{OLLAMA_URL}/api/version", timeout=5).json()["version"]
    print(f"L1 /api/version: {(time.perf_counter() - t0) * 1000:6.0f} мс (версия {version})")

    t0 = time.perf_counter()
    on_disk = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]
    in_ram = httpx.get(f"{OLLAMA_URL}/api/ps", timeout=5).json()["models"]
    print(f"L2 tags + ps:    {(time.perf_counter() - t0) * 1000:6.0f} мс "
          f"(на диске {len(on_disk)}, в памяти {len(in_ram)})")

    for label in ("холодная", "тёплая"):
        t0 = time.perf_counter()
        d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL, "prompt": "Перечисли несколько цветов радуги:",
            "stream": False, "options": {"num_predict": 8, "temperature": 0},
        }, timeout=300).raise_for_status().json()
        print(f"L3 генерация 8 ток ({label}): {time.perf_counter() - t0:5.2f} с "
              f"(загрузка модели {d.get('load_duration', 0) / NS:.2f} с)")
else:
    print("Ollama не запущена — живые ячейки будут пропущены")

У меня: L1 — ~180 мс, L2 — ~330 мс, L3 — ~3 секунды холодная (почти всё —
загрузка модели) и ~1 секунда тёплая. Разница в цене — на порядок: вот почему
уровни разносят по разным интервалам.

И обратите внимание на выбор `num_predict: 8`, а не 1. Мы сначала сделали пробу
на один токен — и получили «скорость» **1 000 000 ток/с**: `eval_duration` одного
токена близок к нулю, деление даёт чушь. Метрика скорости требует хотя бы
нескольких токенов. Ошиблись — починили — записали в урок; это и есть мониторинг.

## Метрики с каждого боевого ответа

Помните урок 1.5: **каждый** ответ Ollama уже несёт телеметрию — `load_duration`,
`prompt_eval_count/duration`, `eval_count/duration`. Не надо гонять отдельные пробы,
чтобы знать скорость продовых запросов, — надо просто **не выбрасывать** эти поля.

Соберём скользящее окно скоростей (тот же приём, что `MetricsWindow` из урока 3.7 —
`deque(maxlen=...)`, перцентили руками). Промпты делаем уникальными: после
prefill-детектива из урока 5.2 мы знаем, что повторный промпт меряет кеш, а не модель:


In [ ]:
class TpsWindow:
    """Скользящее окно скоростей генерации: p50/p95 без внешних библиотек."""

    def __init__(self, maxlen: int = 100):
        self.tps = deque(maxlen=maxlen)
        self.load_seconds = deque(maxlen=maxlen)

    def add_response(self, data: dict) -> None:
        """Вызывайте на КАЖДЫЙ ответ /api/generate|/api/chat — это бесплатно."""
        if data.get("eval_duration"):
            self.tps.append(data["eval_count"] / (data["eval_duration"] / NS))
        self.load_seconds.append(data.get("load_duration", 0) / NS)

    def percentile(self, p: float) -> float | None:
        if not self.tps:
            return None
        ordered = sorted(self.tps)
        return ordered[min(int(len(ordered) * p), len(ordered) - 1)]

    def summary(self) -> str:
        if not self.tps:
            return "нет данных"
        return (f"n={len(self.tps)}  p50={self.percentile(0.50):.1f}  "
                f"p95={self.percentile(0.95):.1f}  min={min(self.tps):.1f} ток/с")


window = TpsWindow()
if OLLAMA_UP:
    for i in range(8):
        d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL,
            "prompt": f"Вопрос {i}: назови {i + 3} любых овоща через запятую.",  # уникальные!
            "stream": False,
            "options": {"temperature": 0, "seed": 42, "num_predict": 40},
        }, timeout=300).raise_for_status().json()
        window.add_response(d)
    print("окно:", ", ".join(f"{t:.0f}" for t in window.tps))
    print("сводка:", window.summary())
else:
    print("Пропущено: Ollama не запущена")

У меня окно ровное: 81–87 ток/с (GPU), p50 ≈ 83. Это и есть **baseline** — эталон
здоровой системы. Сам по себе «83 ток/с» ничего не значит (на CPU нормой были бы
15–17, урок 5.2) — смысл появляется только в сравнении с эталоном этой машины.
Запишите baseline при деплое; без него алерт «медленно» не на чем строить.

## Главная деградация: модель тихо съехала с GPU

Классика жанра на машинах с GPU: видеопамять заняли браузер, игра или вторая
модель — и Ollama молча загружает часть слоёв (или всё) на CPU. Демон жив,
health-check зелёный, ответы правильные. Только в 4–5 раз медленнее.

Диагноз ставится по `/api/ps`: поле `size_vram` против `size` — какая доля модели
живёт в видеопамяти. Устроим учения: baseline у нас есть, теперь «отнимем» GPU
(тот же `num_gpu: 0`, что в уроке 5.2, — честная имитация вытеснения) и посмотрим,
что увидит мониторинг:


In [ ]:
def gpu_share() -> float | None:
    """Доля модели в VRAM: 1.0 — вся на GPU, 0.0 — вся на CPU, None — не загружена."""
    models = httpx.get(f"{OLLAMA_URL}/api/ps", timeout=5).json()["models"]
    if not models:
        return None
    m = models[0]
    return m.get("size_vram", 0) / m.get("size", 1)


if OLLAMA_UP:
    baseline = window.percentile(0.50)
    print(f"baseline: {baseline:.1f} ток/с, доля модели на GPU: {gpu_share():.0%}")

    d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": MODEL, "prompt": "Вопрос X: назови 10 городов через запятую.",
        "stream": False,
        "options": {"temperature": 0, "seed": 42, "num_predict": 40, "num_gpu": 0},
    }, timeout=600).raise_for_status().json()
    tps_now = d["eval_count"] / (d["eval_duration"] / NS)

    print(f"после «вытеснения»: {tps_now:.1f} ток/с, доля на GPU: {gpu_share():.0%}")
    degraded = tps_now < 0.5 * baseline
    print(f"алерт SlowGeneration (tps < 50% baseline): {tps_now:.1f} < {0.5 * baseline:.1f} "
          f"-> {'FIRING' if degraded else 'ok'}")

    unload_model()  # вернуть дефолтный runner (см. «залипание» опций, уроки 5.2-5.3)
else:
    print("Пропущено: Ollama не запущена")

У меня: 83 → 18 ток/с и доля GPU 100% → 0%. Сервис «работает», но в 4.6 раза
медленнее — без метрик это заметят только пользователи, и не сразу. Правило
из урока 4.1 действует и здесь: **самые дорогие сбои — тихие**.

Что смотрит дежурный при жалобе «медленно» (в порядке дешевизны):

| Проверка | Инструмент | Что ищем |
|---|---|---|
| Где модель — GPU или CPU? | `/api/ps` (`size_vram`), `ollama ps` | доля VRAM упала |
| Хост в swap? | `psutil.swap_memory()` | swap > 10% |
| RAM под давлением? | `psutil.virtual_memory()` | available < размер модели |
| Кто-то ест CPU? | диспетчер задач / `psutil.cpu_percent()` | фоновый процесс |
| Контекст раздулся? | `prompt_eval_count` в ответах | prefill стал главной статьёй |

## Метрики хоста: psutil

Этаж 1 — три числа, которые должны сниматься всегда:


In [ ]:
import psutil

GB = 1024**3
mem = psutil.virtual_memory()
swap = psutil.swap_memory()
print(f"RAM: всего {mem.total / GB:.1f} ГБ, доступно {mem.available / GB:.1f} ГБ")
print(f"Swap: занят на {swap.percent:.0f}%")
print(f"CPU: {psutil.cpu_percent(interval=0.5):.0f}% (мгновенный срез)")

# Порог для алерта LowMemory берём не с потолка, а из формулы урока 5.2:
# модели нужно (файл + KV-кеш + runtime); если available меньше — впереди swap или OOM
MODEL_NEEDS_GB = 3.0  # ~для qwen2.5:3b Q4 с контекстом 4k
print(f"\nзапас под модель: {'OK' if mem.available / GB > MODEL_NEEDS_GB else 'МАЛО'} "
      f"(нужно ~{MODEL_NEEDS_GB} ГБ, есть {mem.available / GB:.1f} ГБ)")

## Алерты: правило for-duration

Единичный выброс — не инцидент. Сборщик мусора, случайный пик нагрузки,секундная
пауза диска — если алертить на каждый, дежурный отключит уведомления через день
(и пропустит настоящий пожар). Решение то же, что `for:` в Prometheus-правилах:
**условие должно продержаться N секунд подряд**.

Автомат состояний: `OK -> PENDING (условие видим, ждём) -> FIRING (продержалось) ->
RESOLVED`. Реализуем с инъекцией времени — чтобы протестировать без `time.sleep`
(тот же приём, что тесты на фейках в уроке 3.8):


In [ ]:
class Alert:
    """Алерт с правилом for-duration: OK -> PENDING -> FIRING."""

    def __init__(self, name: str, for_seconds: float):
        self.name = name
        self.for_seconds = for_seconds
        self.violation_since: float | None = None
        self.firing = False

    def update(self, violated: bool, now: float, detail: str = "") -> str:
        if not violated:
            resolved = self.firing
            self.violation_since, self.firing = None, False
            return "RESOLVED" if resolved else "ok"
        if self.violation_since is None:
            self.violation_since = now
        if not self.firing and now - self.violation_since >= self.for_seconds:
            self.firing = True
            return f"FIRING: {detail}"
        return "FIRING (продолжается)" if self.firing else "PENDING"


# Симуляция: свободная RAM по минутам; порог 2 ГБ, for-duration 120 с
alert = Alert("LowMemory", for_seconds=120)
timeline = [(0, 5.0), (60, 1.8), (120, 4.9), (180, 1.7), (240, 1.5), (300, 1.6), (360, 6.0)]
for t, free_gb in timeline:
    state = alert.update(free_gb < 2.0, now=t, detail=f"свободно {free_gb} ГБ")
    print(f"t={t:>3} с  free={free_gb:.1f} ГБ  -> {state}")

Выброс на 60-й секунде остался `PENDING` и не разбудил дежурного; настоящая
проблема (180–300) дошла до `FIRING` через 120 секунд удержания. Цена правила —
время обнаружения: инцидент вы увидите на `for_seconds` позже. Для локальной LLM
разумные значения: OllamaDown — 30–60 с, LowMemory/SwapInUse — 1–2 мин,
SlowGeneration — сразу (проба и так редкая).

## Собираем всё в сторожевой скрипт

`scripts/ollama_watchdog.py` — это весь урок в одном файле: цикл
L1-проверка + psutil каждые `--interval` секунд, редкая глубокая проба
`--probe-model`/`--probe-every`, четыре алерта с for-duration, лог в файл.
Флаг `--max-iterations` добавлен для тестовых прогонов — запустим два витка
прямо отсюда:


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

if OLLAMA_UP:
    log_path = Path(tempfile.gettempdir()) / "ollama_watchdog_demo.log"
    proc = subprocess.run(
        [sys.executable, "scripts/ollama_watchdog.py",
         "--max-iterations", "2", "--interval", "1",
         "--probe-model", MODEL, "--probe-every", "2",
         "--log-file", str(log_path)],
        capture_output=True, text=True, timeout=300, encoding="utf-8", errors="replace",
    )
    print(proc.stderr.strip() or proc.stdout.strip())
    unload_model()  # проба загрузила модель — приберём
else:
    print("Пропущено: Ollama не запущена")

В проде watchdog запускают как службу — теми же способами, что Ollama в уроке 5.3
(NSSM / systemd / Планировщик заданий; решение упражнения 3 — готовые команды).

## Экспорт в Prometheus

Полный стек (Prometheus + правила + Grafana) мы разобрали в уроке 4.3 — здесь
только специфика: **какие** метрики экспортировать для локальной LLM. Мини-экспортер
на prometheus-client — четыре Gauge, обновляемые в цикле watchdog:


In [ ]:
from prometheus_client import CollectorRegistry, Gauge, generate_latest

registry = CollectorRegistry()  # свой реестр, чтобы не мусорить в глобальный
g_up = Gauge("ollama_up", "Демон отвечает на /api/version", registry=registry)
g_tps = Gauge("ollama_gen_tokens_per_second", "p50 скорости генерации по окну", registry=registry)
g_gpu = Gauge("ollama_model_gpu_share", "Доля модели в VRAM (0..1)", registry=registry)
g_ram = Gauge("host_available_ram_bytes", "Доступная RAM хоста", registry=registry)

if OLLAMA_UP:
    g_up.set(1)
    if window.percentile(0.50):
        g_tps.set(window.percentile(0.50))
    g_gpu.set(gpu_share() or 0)
g_ram.set(psutil.virtual_memory().available)

print(generate_latest(registry).decode()[:600])

В реальном сервисе эти строки отдаёт эндпоинт `/metrics` (как в уроках 3.2 и 4.3),
а Prometheus забирает их раз в 15–60 секунд. Правила поверх — с тем самым `for:`:

```yaml
- alert: OllamaDown
  expr: ollama_up == 0
  for: 1m
- alert: ModelFellOffGpu          # наша «тихая» деградация
  expr: ollama_model_gpu_share < 0.9
  for: 2m
- alert: LowMemory
  expr: host_available_ram_bytes < 2e9
  for: 2m
- alert: SlowGeneration           # порог = доля от baseline вашей машины!
  expr: ollama_gen_tokens_per_second < 40
  for: 5m
```

Обратите внимание на порог `SlowGeneration`: 40 — это половина **моего** baseline
(83). На вашей машине число будет другим — порог без baseline бессмыслен.

## ⚠️ Частые ошибки

1. **Гонять глубокую пробу как обычный health-check.** L3 грузит модель, сбрасывает
   keep_alive и занимает слот параллелизма — частые пробы сами создают ту нагрузку,
   которую вы мониторите. L1/L2 — часто, L3 — редко.
2. **Мерить скорость на 1 токене.** `eval_duration` одного токена ≈ 0, и мы получили
   «1 000 000 ток/с» — бессмыслицу. Проба — минимум 8 токенов.
3. **Алерт без baseline.** «18 ток/с — это плохо?» Для CPU-ноутбука — норма, после
   GPU-эталона 83 — деградация в 4.6 раза. Эталон записывается при деплое, пороги —
   доля от него.
4. **Алерт без for-duration.** Каждый выброс будит дежурного → алерты отключают →
   настоящий инцидент пропущен. Единственное исключение — редкие пробы (они сами
   уже «усреднены» интервалом).
5. **Среднее вместо перцентилей.** Мы наступали на это в 3.2 и 4.3: среднее маскирует
   хвосты. Окно + p50/p95.
6. **Тексты запросов в метках метрик.** В Prometheus-метки нельзя класть промпты:
   это ПДн (урок 3.7) и взрыв кардинальности (урок 4.3). Метки — конечные:
   модель, эндпоинт, статус.

## Упражнения

1. **ColdStartStorm.** По окну `load_seconds` из `TpsWindow` напишите детектор
   «модель постоянно выгружается»: алерт, если за последние 10 ответов было ≥ 2
   холодных старта (загрузка > 1 с). Такое бывает, когда keep_alive слишком мал
   или память под давлением и Ollama выгружает модель между запросами.
2. **Детектор «съехали с GPU».** Оформите `gpu_share()` в алерт с for-duration:
   FIRING, если доля < 0.9 дольше двух проверок подряд. Проверьте на синтетической
   таймлинии (как мы делали с LowMemory).
3. **Watchdog как служба.** Настройте автозапуск `ollama_watchdog.py` на вашей ОС:
   Планировщик заданий Windows или systemd-таймер. Подсказка: команды — в решении.

Решения — ниже.


## Мини-квиз

<details><summary><b>1. Почему глубокую пробу (L3) нельзя гонять каждые 10 секунд, как /api/version?</b></summary>

Она деструктивна: загружает модель в память (отбирая её у других), сбрасывает
таймер keep_alive и занимает слот параллелизма — при `NUM_PARALLEL=1` реальный
запрос будет ждать позади пробы. Наблюдатель влияет на систему; частые L3-пробы —
это самостоятельная нагрузка.
</details>

<details><summary><b>2. Проба вернула скорость 1 000 000 ток/с. Что произошло?</b></summary>

Проба генерировала 1 токен: `eval_duration` при этом почти нулевой, и деление
даёт бессмыслицу. Метрика скорости требует нескольких токенов (мы поставили 8).
Отдельный урок: не доверяйте метрике, пока не видели её на здоровой системе.
</details>

<details><summary><b>3. Демон жив, health-check зелёный, пользователи жалуются на скорость. Первые две проверки?</b></summary>

`/api/ps` — какая доля модели в VRAM (`size_vram` против `size`): классика —
модель тихо съехала на CPU. И `psutil.swap_memory()` — не ушёл ли хост в swap.
Обе проверки бесплатны и отвечают за 90% случаев «вдруг стало медленно».
</details>

<details><summary><b>4. Чем плох алерт «средняя скорость за час ниже X»?</b></summary>

Дважды: среднее маскирует хвосты (десять быстрых запросов скроют один
десятисекундный — уроки 3.2 и 4.3), а фиксированный X без baseline машины
неинтерпретируем. Правильно: перцентили по скользящему окну + порог как доля
от эталона, записанного при деплое.
</details>

<details><summary><b>5. Что мы выигрываем и что проигрываем, ставя for-duration = 2 минуты?</b></summary>

Выигрываем тишину: единичные выбросы (GC, пик диска, случайный тяжёлый запрос)
не будят дежурного. Проигрываем время обнаружения: настоящий инцидент увидим
на 2 минуты позже. Это осознанный размен, а не бесплатный фильтр.
</details>


## Итоги

- Локальная LLM мониторится на трёх этажах: хост (psutil: RAM, swap), демон
  (`/api/version`, `/api/ps`), качество (скорость, холодные старты).
- Health-check — пирамида: L1/L2 дешёвые и частые, L3 (генерация) — деструктивная
  и редкая. Минимум 8 токенов в пробе.
- Скорость собирается бесплатно с каждого боевого ответа (телеметрия Ollama) —
  окно + p50/p95 + baseline машины.
- Главная тихая деградация — модель съехала с GPU: ловится по `size_vram`
  в `/api/ps` (у меня: 83 → 18 ток/с при доле GPU 100% → 0%).
- Алерты — с for-duration (OK → PENDING → FIRING), пороги — доля от baseline.
  Всё вместе — `scripts/ollama_watchdog.py`; экспорт — четыре Gauge в Prometheus.

**Дальше**: урок 5.6 — когда алерт уже горит: OOM, зависания, конкуренция,
версии моделей — диагностика и лечение.

## Полезные ссылки

- psutil: <https://psutil.readthedocs.io>
- prometheus-client: <https://prometheus.github.io/client_python/>
- Правила алертинга Prometheus: <https://prometheus.io/docs/prometheus/latest/configuration/alerting_rules/>
- Поля ответа Ollama: <https://github.com/ollama/ollama/blob/main/docs/api.md>


---

## Решения упражнений

### Упражнение 1: ColdStartStorm


In [ ]:
def cold_start_storm(load_seconds: deque, threshold_s: float = 1.0,
                     window_n: int = 10, max_cold: int = 1) -> tuple[bool, str]:
    """Алерт: слишком много холодных стартов в последних window_n ответах."""
    recent = list(load_seconds)[-window_n:]
    cold = sum(1 for s in recent if s > threshold_s)
    fired = cold > max_cold
    return fired, f"{cold} холодных стартов из {len(recent)} последних"


# Синтетика: здоровая полоса, потом модель начинает выгружаться между запросами
healthy = deque([0.01] * 10)
sick = deque([0.01, 2.9, 0.01, 0.01, 3.1, 0.02, 2.8, 0.01, 0.01, 3.0])
for name, series in [("здоровое окно", healthy), ("больное окно", sick)]:
    fired, detail = cold_start_storm(series)
    print(f"{name}: {'FIRING' if fired else 'ok'} ({detail})")
# Лечение — урок 5.3: поднять keep_alive, проверить давление на память
# (кто-то другой ест RAM -> Ollama выгружает модель), MAX_LOADED_MODELS.

### Упражнение 2: детектор «съехали с GPU»


In [ ]:
gpu_alert = Alert("ModelFellOffGpu", for_seconds=2)  # «2 секунды» = 2 проверки при шаге 1

# Синтетическая таймлиния долей VRAM (1.0 = вся модель на GPU)
shares = [(0, 1.0), (1, 1.0), (2, 0.4), (3, 0.4), (4, 0.0), (5, 1.0)]
for t, share in shares:
    state = gpu_alert.update(share < 0.9, now=t, detail=f"на GPU только {share:.0%}")
    print(f"t={t}  gpu_share={share:.0%}  -> {state}")

# Живая проверка текущего состояния (модель мы выгрузили — доли может не быть):
if OLLAMA_UP:
    share = gpu_share()
    print("\nсейчас:", "модель не загружена" if share is None else f"доля на GPU {share:.0%}")

### Упражнение 3: watchdog как служба

**Windows (Планировщик заданий)** — задача при старте системы, от имени пользователя:

```powershell
schtasks /Create /TN "OllamaWatchdog" /SC ONSTART /RU $env:USERNAME `
  /TR "C:\путь\к\venv\Scripts\python.exe C:\...\scripts\ollama_watchdog.py --probe-model qwen2.5:3b"
```

(Или через NSSM — как саму Ollama в уроке 5.3: тогда будут рестарты при падении.)

**Linux (systemd)** — юнит + автоперезапуск:

```ini
[Unit]
Description=Ollama watchdog
After=ollama.service

[Service]
ExecStart=/opt/venv/bin/python /opt/llm/scripts/ollama_watchdog.py --probe-model qwen2.5:3b
Restart=always
RestartSec=10

[Install]
WantedBy=multi-user.target
```

Ирония, которую стоит увидеть: watchdog — тоже сервис, и его тоже надо
перезапускать при падении. Кто сторожит сторожа — супервизор ОС.
